# Dataset 

Dataset columns : 

train_columns = ['user_id', 'item_seq', 'item_id', 'likes_level', 'views_level', 'label']

test_columns = ['user_id', 'item_seq', 'item_id', 'likes_level', 'views_level', 'label']

valid_columns = ['user_id', 'item_seq', 'item_id', 'likes_level', 'views_level', 'label']

item_feature_columns = [
    'item_id', 'item_title', 'item_tags', 'likes_level', 'views_level',
    'txt_emb_BERT', 'img_emb_CLIPRN50'
]

item_info_columns = ['item_id', 'item_tags', 'item_emb_d128']

item_emb_columns = [
    'item_id', 'item_emb_d128_v1', 'item_emb_d128_v2',
    'item_emb_d128_v3', 'item_emb_d128_e4'
]

item_seq_columns = ['user_id', 'item_seq']


# Install dependencies

In [ ]:
!pip install --force-reinstall --no-cache-dir \
    numpy==1.26.4 \
    pandas==2.2.2 \
    scikit-learn==1.4.0 \
    scipy==1.11.4 \
    torch torchvision torchaudio

In [ ]:
!pip install fuxictr==2.3.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.8 MB/s eta 0:00:00


# Copy from datasets the repo dir

In [ ]:
!cp -r /kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/* /kaggle/working/data

# Extract image embeddings using CLIP-Large

In [ ]:
!pip install -q transformers torch pandas pillow tqdm

import os
import torch
import pandas as pd
import numpy as np
import os
import glob
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm
import gc

In [ ]:
# CONFIGURATION
class Config:
    # Input Paths
    IMAGES_DIR = '/kaggle/working/WWW2025_MMCTR/data/item_images/item_images' 
    FEATURE_PATH = '/kaggle/working/WWW2025_MMCTR/data/item_feature.parquet'
    
    # Checkpoint/Output Directory (Create a dataset or persistent dir if possible)
    # In Kaggle, /kaggle/working is temporary but survives session restarts if "Persistence" is on? 
    # Usually better to rely on manual downloading or dataset mounting for long jobs.
    OUTPUT_DIR = '/kaggle/working/EMB_GEN_CLIP/embeddings_parts' 
    FINAL_FILENAME = 'item_emb_clip_large_full.parquet'
    
    # Model
    MODEL_NAME = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K" 
    
    # Performance
    BATCH_SIZE = 32
    NUM_WORKERS = 4
    SAVE_EVERY = 5000 # Save a chunk every 5000 items
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# MULTI-GPU WRAPPER
class CLIPMultiGPU(nn.Module):
    """
    Wrapper to ensure DataParallel returns tensors nicely.
    DataParallel scatters inputs and gathers outputs. 
    We need strict tensor outputs for gather to work.
    """
    def __init__(self, model_name):
        super().__init__()
        self.model = CLIPModel.from_pretrained(model_name)
        
    def forward(self, input_ids, pixel_values, attention_mask):
        # Get features
        img_features = self.model.get_image_features(pixel_values)
        txt_features = self.model.get_text_features(input_ids, attention_mask=attention_mask)
        
        # Normalize (L2) - Crucial for CLIP
        img_features = img_features / img_features.norm(p=2, dim=-1, keepdim=True)
        txt_features = txt_features / txt_features.norm(p=2, dim=-1, keepdim=True)
        
        return img_features, txt_features

# DATASET
class MMCTRDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.item_ids = df['item_id'].values
        self.titles = df['item_title'].fillna("").astype(str).values
        self.img_dir = img_dir
        self.processor = processor
        
    def __len__(self):
        return len(self.item_ids)
    
    def __getitem__(self, idx):
        item_id = self.item_ids[idx]
        text = self.titles[idx]
        
        # Image Loading Logic
        img_path = os.path.join(self.img_dir, f"{item_id}.jpg")
            
        if os.path.exists(img_path):
            try:
                image = Image.open(img_path).convert("RGB")
            except:
                image = Image.new("RGB", (224, 224), (0, 0, 0))
                print(f"Error loading {img_path} Using black image as placeholder")
        else:
            image = Image.new("RGB", (224, 224), (0, 0, 0))
            print(f"Error loading {img_path} Using black image as placeholder")
            
        inputs = self.processor(
            text=[text], 
            images=image, 
            return_tensors="pt", 
            padding='max_length',
            truncation=True,
            max_length=77
        )
        
        return {
            "item_id": str(item_id), 
            "pixel_values": inputs.pixel_values.squeeze(0),
            "input_ids": inputs.input_ids.squeeze(0),
            "attention_mask": inputs.attention_mask.squeeze(0)
        }

# UTILS
def get_completed_ids(output_dir):
    """Scans the output directory for existing parts and returns processed item_ids."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        return set()
    
    files = glob.glob(f"{output_dir}/part_*.parquet")
    completed_ids = set()
    print(f"Found {len(files)} existing parts. Scanning for completed items...")
    
    for f in files:
        try:
            df = pd.read_parquet(f, columns=['item_id'])
            completed_ids.update(df['item_id'].astype(str).tolist())
        except Exception as e:
            print(f"Warning: Could not read {f}. Skipping.")
            
    return completed_ids

def save_chunk(buffer_ids, buffer_embs, output_dir, chunk_idx):
    """Saves a chunk of data to parquet."""
    df_chunk = pd.DataFrame({
        'item_id': buffer_ids,
        'clip_large_emb': list(buffer_embs)
    })
    save_path = f"{output_dir}/part_{chunk_idx}.parquet"
    df_chunk.to_parquet(save_path, index=False)
    print(f"Saved chunk {chunk_idx} ({len(df_chunk)} items)")

# MAIN
def run_extraction():
    # Load Data
    print("Loading Metadata...")
    df_all = pd.read_parquet(Config.FEATURE_PATH)
    total_items = len(df_all)
    
    # Check for Resume
    completed_ids = get_completed_ids(Config.OUTPUT_DIR)
    print(f"Already processed: {len(completed_ids)} / {total_items}")
    
    # Filter DataFrame
    df_all['item_id'] = df_all['item_id'].astype(str)
    df_todo = df_all[~df_all['item_id'].isin(completed_ids)].reset_index(drop=True)
    
    if len(df_todo) == 0:
        print("All items processed! Proceeding to merge...")
    else:
        print(f"Remaining items to process: {len(df_todo)}")
        
        # Model Setup (Multi-GPU)
        print(f"Loading Model on {torch.cuda.device_count()} GPUs...")
        model = CLIPMultiGPU(Config.MODEL_NAME).to(Config.DEVICE)
        processor = CLIPProcessor.from_pretrained(Config.MODEL_NAME)
        
        if torch.cuda.device_count() > 1:
            print("Activating DataParallel...")
            model = nn.DataParallel(model)
        
        model.eval()
        
        # DataLoader
        dataset = MMCTRDataset(df_todo, Config.IMAGES_DIR, processor)
        loader = DataLoader(
            dataset, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=False, 
            num_workers=Config.NUM_WORKERS,
            pin_memory=True
        )
        
        # Extraction Loop
        buffer_embs = []
        buffer_ids = []
        chunk_counter = len(glob.glob(f"{Config.OUTPUT_DIR}/part_*.parquet"))
        
        print("Starting Extraction...")
        with torch.no_grad():
            for batch in tqdm(loader):
                # Inputs to GPU
                pixel_values = batch['pixel_values'].to(Config.DEVICE)
                input_ids = batch['input_ids'].to(Config.DEVICE)
                attention_mask = batch['attention_mask'].to(Config.DEVICE)
                item_ids = batch['item_id'] # Keep on CPU
                
                # Forward Pass (Handled by DataParallel)
                img_emb, txt_emb = model(input_ids, pixel_values, attention_mask)
                
                # Concatenate & Move to CPU
                # shape: (B, 768) + (B, 768) -> (B, 1536)
                batch_embs = torch.cat([img_emb, txt_emb], dim=1).cpu().numpy().astype(np.float16) # float16 saves space
                
                buffer_embs.extend(batch_embs)
                buffer_ids.extend(item_ids)
                
                # Checkpoint Save
                if len(buffer_ids) >= Config.SAVE_EVERY:
                    save_chunk(buffer_ids, buffer_embs, Config.OUTPUT_DIR, chunk_counter)
                    buffer_ids = []
                    buffer_embs = []
                    chunk_counter += 1
                    gc.collect()

            # Save remaining buffer
            if len(buffer_ids) > 0:
                save_chunk(buffer_ids, buffer_embs, Config.OUTPUT_DIR, chunk_counter)

    

if __name__ == "__main__":
    run_extraction()

In [ ]:
import pyarrow.dataset as ds
import glob

# Final Merge
print("Merging all parts into final file...")
all_files = glob.glob(f"{Config.OUTPUT_DIR}/part_*.parquet")
if len(all_files) > 0:
    # Read all parts and concat
    # Note: If dataset is huge, merging in memory might crash. 
    # If it crashes here, you can skip this and just use the parts.
    try:
        dataset = ds.dataset(Config.OUTPUT_DIR, format="parquet")
        df_final = dataset.to_table().to_pandas()
        df_final.to_parquet(Config.FINAL_FILENAME, index=False)
        print(f"Success! Saved full dataset to {Config.FINAL_FILENAME}")
    except Exception as e:
        print("Merge in memory failed (RAM issue?)")
        print(e)
else:
    print("No files found to merge.")

# Contrastive fine-tuning for training an MLP that generate 128d embeddings from 2048d embedding of clip (GPU-Optimized dataloading)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GPUProjectionNetwork(nn.Module):
    def __init__(self, pretrained_matrix, input_dim=2048, hidden_dim=512, output_dim=128):
        super().__init__()
        # The Frozen Matrix (Index 0 is padding)
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_matrix, 
            freeze=True, 
            padding_idx=0 
        )
        
        # The Trainable Projector
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, hist_idx, pos_idx, neg_idx=None):
        # Look up vectors (B, SeqLen, 2048)
        hist_vecs = self.embedding(hist_idx)
        pos_vec = self.embedding(pos_idx)
        
        # Mean Pooling (Ignoring Padding at Index 0)
        # Create a mask: 1 where data exists, 0 where padding (index 0)
        mask = (hist_idx != 0).unsqueeze(-1).float() # (B, SeqLen, 1)
        
        # Sum vectors and divide by count of non-padding items
        sum_vecs = (hist_vecs * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1) # Avoid div by zero
        hist_mean = sum_vecs / counts
        
        # C. Project and Normalize
        hist_proj = F.normalize(self.net(hist_mean), p=2, dim=1)
        pos_proj = F.normalize(self.net(pos_vec), p=2, dim=1)
        
        if neg_idx is not None:
            neg_vec = self.embedding(neg_idx)
            neg_proj = F.normalize(self.net(neg_vec), p=2, dim=1)
            return hist_proj, pos_proj, neg_proj
        
        return hist_proj, pos_proj

In [ ]:
# OPTIMIZED DATASET (Returns Integers for GPU Lookup)
from torch.utils.data import Dataset
import numpy as np

class FastUserSeqDataset(Dataset):
    def __init__(self, seq_df, id_to_index_map, max_history=5):
        self.seqs = seq_df['item_seq'].tolist()
        self.id_map = id_to_index_map
        self.all_items = list(id_to_index_map.keys())
        self.max_history = max_history
        self.pad_idx = 0 # Explicitly use 0 for padding
        
    def __len__(self):
        return len(self.seqs)
    
    def __getitem__(self, idx):
        raw_seq = self.seqs[idx]
        real_seq = [x for x in raw_seq if x != 0] # Remove padding id (0)
        
        # 1. Identify Target & History
        if len(real_seq) < 2:
            if len(real_seq) == 1:
                target_id = real_seq[0]
                hist_ids = [real_seq[0]]
            else:
                target_id = self.all_items[0] # Safety fallback
                hist_ids = [target_id]
        else:
            target_id = real_seq[-1]
            hist_ids = real_seq[:-1][-self.max_history:]
            
        # 2. Convert to Indices (using .get with default 0)
        target_idx = self.id_map.get(str(target_id), self.pad_idx)
        hist_indices = [self.id_map.get(str(h), self.pad_idx) for h in hist_ids]
        
        # 3. Pad History with 0s at the start
        if len(hist_indices) < self.max_history:
            pads = [self.pad_idx] * (self.max_history - len(hist_indices))
            hist_indices = pads + hist_indices
            
        # 4. Negative Sampling (Random Index, avoid 0)
        # Assumes len(id_map) maps to max index N. Valid range [1, N]
        neg_idx = np.random.randint(1, len(self.id_map))
        
        return {
            'hist_idx': torch.tensor(hist_indices, dtype=torch.long),
            'pos_idx': torch.tensor(target_idx, dtype=torch.long),
            'neg_idx': torch.tensor(neg_idx, dtype=torch.long)
        }

In [ ]:
import pandas as pd
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import os
import gc

class Config:
    # PATHS
    EMB_PATH = '/kaggle/working/item_emb_clip_large_full.parquet'
    SEQ_PATH = '/kaggle/working/WWW2025_MMCTR_Challenge/data/item_seq.parquet'
    OUTPUT_PATH = '/kaggle/working/CFT-EMB-128/item_emb_d128_cft.parquet'

    # Checkpoint Configuration
    CHECKPOINT_DIR = '/kaggle/working/CFT-EMB-128/checkpoints'
    CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'projector_ckpt_gpu_opt_latest.pth')
    SAVE_EVERY_STEPS = 50
    
    HISTORY_WINDOW = 5
    
    # DIMENSIONS
    # Input is 2048 because we concat Image(1024) + Text(1024) from ViT-H
    INPUT_DIM = 2048 
    HIDDEN_DIM = 512  # The "Digest" layer
    OUTPUT_DIM = 128  # The Challenge Requirement
    
    # TRAINING
    BATCH_SIZE = 2048 # Bigger batch = better contrastive learning
    EPOCHS = 10       # Usually converges quickly (3-5 epochs often enough)
    LR = 1e-3         # Standard Adam learning rate
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    SEED = 42

def train():
    print(f"Device: {Config.DEVICE} | GPU Count: {torch.cuda.device_count()}")
    torch.manual_seed(Config.SEED)
    np.random.seed(Config.SEED)
    
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

    # --- PART A: PREPARE MATRIX (Index 0 = Padding) ---
    print("1. Preparing Matrix...")
    df_emb = pd.read_parquet(Config.EMB_PATH)
    col = 'clip_large_emb' if 'clip_large_emb' in df_emb.columns else df_emb.columns[1]
    
    # Map ItemID -> Index (Start from 1)
    all_ids = df_emb['item_id'].astype(str).values
    id_to_index = {oid: i for i, oid in enumerate(all_ids, start=1)}
    
    # Stack Vectors & Prepend Zero Row
    vectors = np.stack(df_emb[col].values)
    padding_row = np.zeros((1, Config.INPUT_DIM), dtype=vectors.dtype)
    matrix_with_pad = np.concatenate([padding_row, vectors], axis=0)
    
    pretrained_tensor = torch.tensor(matrix_with_pad, dtype=torch.float32)
    print(f"   Matrix Shape: {pretrained_tensor.shape} (Row 0 is Padding)")
    
    del df_emb, vectors, matrix_with_pad, padding_row
    gc.collect()

    # --- PART B: SETUP ---
    df_seq = pd.read_parquet(Config.SEQ_PATH)
    dataset = FastUserSeqDataset(df_seq, id_to_index, max_history=Config.HISTORY_WINDOW)
    
    model = GPUProjectionNetwork(pretrained_tensor).to(Config.DEVICE)
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
        
    optimizer = optim.Adam(model.parameters(), lr=Config.LR)
    criterion = nn.TripletMarginLoss(margin=1.0, p=2)

    # --- PART C: RESUME LOGIC ---
    start_epoch = 0
    start_batch_idx = 0
    running_loss_sum = 0
    running_batch_count = 0
    
    if os.path.exists(Config.CHECKPOINT_FILE):
        print(f"Loading checkpoint...")
        ckpt = torch.load(Config.CHECKPOINT_FILE, map_location=Config.DEVICE)
        
        if isinstance(model, nn.DataParallel):
            try: model.module.load_state_dict(ckpt['model_state_dict'])
            except: model.load_state_dict(ckpt['model_state_dict'])
        else:
            model.load_state_dict(ckpt['model_state_dict'])
            
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch']
        start_batch_idx = ckpt.get('batch_idx', -1) + 1
        
        # Load running loss if resuming mid-epoch
        if ckpt.get('running_loss_sum') is not None:
            running_loss_sum = ckpt['running_loss_sum']
            running_batch_count = ckpt['running_batch_count']
            print(f"Resuming Epoch {start_epoch+1}, Batch {start_batch_idx}")

    # --- PART D: TRAINING LOOP ---
    if start_epoch < Config.EPOCHS:
        model.train()
        for epoch in range(start_epoch, Config.EPOCHS):
            # Reset running loss if starting a FRESH epoch (not resuming)
            if epoch != start_epoch:
                running_loss_sum = 0
                running_batch_count = 0
                
            # Efficient Shuffle/Skip Logic
            g = torch.Generator()
            g.manual_seed(Config.SEED + epoch)
            perm_indices = torch.randperm(len(dataset), generator=g)
            
            offset = 0
            if epoch == start_epoch and start_batch_idx > 0:
                offset = start_batch_idx * Config.BATCH_SIZE
                if offset < len(perm_indices):
                    perm_indices = perm_indices[offset:]
                else:
                    continue # Epoch done
            
            subset = torch.utils.data.Subset(dataset, perm_indices)
            loader = DataLoader(subset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4)
            
            pbar = tqdm(enumerate(loader), total=len(loader), desc=f"Ep {epoch+1}")
            
            for local_i, batch in pbar:
                real_idx = start_batch_idx + local_i if epoch == start_epoch else local_i
                
                optimizer.zero_grad()
                
                # Inputs are Integers
                h = batch['hist_idx'].to(Config.DEVICE)
                p = batch['pos_idx'].to(Config.DEVICE)
                n = batch['neg_idx'].to(Config.DEVICE)
                
                # Model handles lookup
                h_proj, p_proj, n_proj = model(h, p, n)
                
                loss = criterion(h_proj, p_proj, n_proj)
                loss.backward()
                optimizer.step()
                
                # Stats
                running_loss_sum += loss.item()
                running_batch_count += 1
                avg = running_loss_sum / running_batch_count
                pbar.set_description(f"Ep {epoch+1} | Batch {real_idx} | Ep AvgLoss: {avg:.4f}")
                
                # Intra-Epoch Save
                if (real_idx + 1) % Config.SAVE_EVERY_STEPS == 0:
                    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                    torch.save({
                        'epoch': epoch,
                        'batch_idx': real_idx,
                        'model_state_dict': state,
                        'optimizer_state_dict': optimizer.state_dict(),
                        'running_loss_sum': running_loss_sum,
                        'running_batch_count': running_batch_count
                    }, Config.CHECKPOINT_FILE)
            
            # End Epoch Save
            start_batch_idx = 0 
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save({
                'epoch': epoch + 1,
                'batch_idx': -1,
                'model_state_dict': state,
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': running_loss_sum / running_batch_count
            }, Config.CHECKPOINT_FILE)

    print("Training Finished!")

if __name__ == "__main__":
    train()

# 128d embedding generation (gpu-optimized)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import os
import gc


class GPUProjectionNetwork(nn.Module):
    def __init__(self, pretrained_matrix, input_dim=2048, hidden_dim=512, output_dim=128):
        super().__init__()
        # The Frozen Matrix (Index 0 is padding)
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_matrix, 
            freeze=True, 
            padding_idx=0 
        )
        
        # The Trainable Projector
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, hist_idx, pos_idx, neg_idx=None):
        # Look up vectors (B, SeqLen, 2048)
        hist_vecs = self.embedding(hist_idx)
        pos_vec = self.embedding(pos_idx)
        
        # Mean Pooling (Ignoring Padding at Index 0)
        # Create a mask: 1 where data exists, 0 where padding (index 0)
        mask = (hist_idx != 0).unsqueeze(-1).float() # (B, SeqLen, 1)
        
        # Sum vectors and divide by count of non-padding items
        sum_vecs = (hist_vecs * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1) # Avoid div by zero
        hist_mean = sum_vecs / counts
        
        # Project and Normalize
        hist_proj = F.normalize(self.net(hist_mean), p=2, dim=1)
        pos_proj = F.normalize(self.net(pos_vec), p=2, dim=1)
        
        if neg_idx is not None:
            neg_vec = self.embedding(neg_idx)
            neg_proj = F.normalize(self.net(neg_vec), p=2, dim=1)
            return hist_proj, pos_proj, neg_proj
        
        return hist_proj, pos_proj
        
# CONFIGURATION
class Config:
    
    EMB_PATH = '/kaggle/working/item_emb_clip_large_full.parquet' 
    CHECKPOINT_FILE = '/kaggle/working/CFT-EMB-128/checkpoints/projector_ckpt_latest.pth'
    OUTPUT_PATH = '/kaggle/working/CFT-EMB-128/item_emb_d128.parquet' 
    
    INPUT_DIM = 2048 
    HIDDEN_DIM = 512
    OUTPUT_DIM = 128
    
    CHUNK_SIZE = 5000
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# INFERENCE ROUTINE
def generate_embeddings_only():
    print(f"Device: {Config.DEVICE}")
    
    # Load Model Structure (With Dummy Matrix)
    print("Initializing Model...")
    # We pass a tiny dummy matrix because we only care about the MLP (self.net)
    dummy_matrix = torch.zeros((1, Config.INPUT_DIM))
    model = GPUProjectionNetwork(dummy_matrix).to(Config.DEVICE)
    
    # Load Weights from Checkpoint
    if not os.path.exists(Config.CHECKPOINT_FILE):
        print(f"Error: Checkpoint file {Config.CHECKPOINT_FILE} not found!")
        return

    print(f"Loading weights from {Config.CHECKPOINT_FILE}...")
    checkpoint = torch.load(Config.CHECKPOINT_FILE, map_location=Config.DEVICE)
    state_dict = checkpoint['model_state_dict']

    # --- FIX FOR DATA PARALLEL & FILTERING ---
    new_state_dict = {}
    for k, v in state_dict.items():
        # 1. Strip 'module.' prefix if present
        name = k[7:] if k.startswith('module.') else k
        
        # 2. FILTER OUT EMBEDDING LAYER
        # We don't want to load the massive lookup matrix from training
        # We only want the MLP weights ('net.0.weight', etc.)
        if 'embedding' not in name:
            new_state_dict[name] = v
            
    # Load with strict=False because we are intentionally skipping 'embedding.weight'
    model.load_state_dict(new_state_dict, strict=False)
    model.eval()
    print("Model loaded successfully (MLP weights only).")

    # 3. Load Data
    print(f"Loading input embeddings from {Config.EMB_PATH}...")
    try:
        df_emb = pd.read_parquet(Config.EMB_PATH)
    except FileNotFoundError:
        print("Input file not found.")
        return

    col_name = 'clip_large_emb' if 'clip_large_emb' in df_emb.columns else df_emb.columns[1]
    print(f"Using column: {col_name}")

    all_ids = df_emb['item_id'].tolist()
    all_embs_np = np.stack(df_emb[col_name].values) 
    
    print(f"Data Shape: {all_embs_np.shape}")
    
    # 4. Run Inference in Chunks
    final_embs = []
    
    print("Projecting embeddings...")
    with torch.no_grad():
        for i in tqdm(range(0, len(all_embs_np), Config.CHUNK_SIZE)):
            chunk_np = all_embs_np[i : i + Config.CHUNK_SIZE]
            chunk_tensor = torch.tensor(chunk_np, dtype=torch.float32).to(Config.DEVICE)
            
            # --- CRITICAL CHANGE ---
            # We bypass model.forward() because that expects Indices.
            # We call model.net() directly because we have Vectors.
            output = model.net(chunk_tensor)
            
            # Manually normalize (since forward() usually does this)
            output = F.normalize(output, p=2, dim=1)
            
            final_embs.append(output.cpu().numpy())
            
    final_embs = np.concatenate(final_embs, axis=0)
    
    # 5. Save Result
    print(f"Saving {len(final_embs)} embeddings to {Config.OUTPUT_PATH}...")
    df_out = pd.DataFrame({
        'item_id': all_ids,
        'item_emb_d128': list(final_embs)
    })
    
    df_out.to_parquet(Config.OUTPUT_PATH, index=False)
    print("Done! File ready.")

if __name__ == "__main__":
    generate_embeddings_only()

## Copy the generated embeddings to item_info

In [ ]:
import pandas as pd
import numpy as np

# 1. SETUP PATHS
DATA_DIR = '/kaggle/working/data/MicroLens_1M_x1'
MY_EMB_PATH = '/kaggle/working/CFT-EMB-128/item_emb_d128.parquet'
ORIGINAL_INFO_PATH = f'{DATA_DIR}/item_info.parquet'
BACKUP_PATH = f'{DATA_DIR}/item_info_original.parquet'

# 2. LOAD FILES
print("Loading data...")
df_info = pd.read_parquet(ORIGINAL_INFO_PATH)
df_my_emb = pd.read_parquet(MY_EMB_PATH)

# 3. BACKUP ORIGINAL
if not os.path.exists(BACKUP_PATH):
    print(f"Backing up original item_info to {BACKUP_PATH}...")
    df_info.to_parquet(BACKUP_PATH, index=False)

# 4. PREPARE MERGE
# Ensure column names match. 
# The target file expects a column named 'item_emb'
target_col = 'item_emb'
# Rename your column to match target
if 'item_emb_d128' in df_my_emb.columns:
    df_my_emb = df_my_emb.rename(columns={'item_emb_d128': target_col})

# 5. INJECT (Merge & Overwrite)
print("Injecting new embeddings...")

# We force your df_my_emb ID to match the type of the original df_info ID
target_type = df_info['item_id'].dtype
print(f"Aligning item_id types to: {target_type}")

df_my_emb['item_id'] = df_my_emb['item_id'].astype(target_type)

# Drop the old embedding column from df_info to avoid duplicates
if target_col in df_info.columns:
    df_info = df_info.drop(columns=[target_col])

# Merge on item_id
# 'left' merge ensures we keep all items in item_info, even if your file missed some
df_merged = pd.merge(df_info, df_my_emb[['item_id', target_col]], on='item_id', how='left')

# 6. FILL MISSING (Critical)
# If your training skipped some items (rare), fill them with zeros to prevent crash
null_mask = df_merged[target_col].isnull()
if null_mask.sum() > 0:
    print(f"Warning: {null_mask.sum()} items missing in the embedding file. Filling with zeros.")
    # Create zero vector of size 128
    zero_vec = np.zeros(128, dtype=np.float32).tolist()
    df_merged.loc[null_mask, target_col] = pd.Series([zero_vec] * null_mask.sum()).values

# 7. SAVE
print(f"Saving modified data to {ORIGINAL_INFO_PATH}...")
df_merged.to_parquet(ORIGINAL_INFO_PATH, index=False)
print("Done refactoring item_info.parquet with new embeddings!")

# Train the CTR model

## Create dataset and model configuration files

In [ ]:
config="""
MicroLens_1M_x1:
    data_format: parquet
    data_root: ./data/
    feature_cols:
    - {active: true, dtype: int, name: user_id, type: meta}
    - {active: true, dtype: int, name: item_seq, type: meta}
    - {active: true, dtype: int, name: likes_level, type: categorical, vocab_size: 11}
    - {active: true, dtype: int, name: views_level, type: categorical, vocab_size: 11}
    - {active: true, dtype: int, name: item_id, source: item, type: categorical, vocab_size: 91718}
    - {active: true, dtype: int, max_len: 5, name: item_tags, source: item, type: sequence,
        vocab_size: 11740}
    - {active: true, dtype: float, embedding_dim: 128, name: item_emb_d128, source: item,
        type: embedding}
    item_info: ./data/MicroLens_1M_x1/item_info.parquet
    label_col: {dtype: float, name: label}
    rebuild_dataset: false
    test_data: ./data/MicroLens_1M_x1/test.parquet
    train_data: ./data/MicroLens_1M_x1/train.parquet
    valid_data: ./data/MicroLens_1M_x1/valid.parquet
"""

with open("config/dataset_config/MicroLens_dataset.yaml", "w") as f:
    f.write(config)

In [ ]:
config="""
Transformer_DCN_MicroLens:
  gpu: 0
  model: Transformer_DCN
  dataset_id: MicroLens_1M_x1
  optimizer: Adam
  learning_rate: 0.001
  loss: binary_crossentropy
  batch_size: 1024
  epochs: 7
  metrics: ['AUC', 'logloss']
  monitor: 'AUC'
  monitor_mode: max
  early_stop_patience: 5
  shuffle: true
  seed: 20242025
  verbose: 1
  embedding_dim: 64
  num_heads: 4
  transformer_layers: 3
  dim_feedforward: 512
  transformer_dropout: 0.1
  dcn_cross_layers: 3
  dcn_hidden_units: [512, 256, 128]
  mlp_hidden_units: [64, 32]
  batch_norm: true
  net_dropout: 0.2
  first_k_cols: 16
  group_id: user_id
  concat_max_pool: true
  embedding_regularizer: 0
  net_regularizer: 0
  hidden_activations: relu
  task: binary_classification
  model_root: ./checkpoints/
  save_best_only: true
  pickle_feature_encoder: true
  num_workers: 4
  eval_steps: null
  feature_config: null
  feature_specs: null
  use_features: null
  debug_mode: false
"""

with open("config/model_config/transformer_dcn_config.yaml", "w") as f:
    f.write(config)

## Create model architecture

In [ ]:
import torch
from fuxictr.utils import not_in_whitelist
from torch import nn
import random
from fuxictr.pytorch.models import BaseModel
from fuxictr.pytorch.layers import FeatureEmbedding, MLP_Block, CrossNetV2


class Transformer_DCN(BaseModel):
    def __init__(self,
                 feature_map,
                 model_id="Transformer_DCN",
                 gpu=-1,
                 hidden_activations="ReLU",
                 dcn_cross_layers=3,
                 dcn_hidden_units=[1024, 512, 256],
                 mlp_hidden_units=[64, 32],
                 num_heads=1,
                 transformer_layers=2,
                 transformer_dropout=0.2,
                 dim_feedforward=256,
                 learning_rate=5e-4,
                 embedding_dim=64,
                 net_dropout=0.2,
                 first_k_cols=16,
                 batch_norm=False,
                 concat_max_pool=True,
                 accumulation_steps=1,
                 embedding_regularizer=None,
                 net_regularizer=None,
                 **kwargs):
        super().__init__(feature_map,
                         model_id=model_id,
                         gpu=gpu,
                         embedding_regularizer=embedding_regularizer,
                         net_regularizer=net_regularizer,
                         **kwargs)
        self.feature_map = feature_map
        self.embedding_dim = embedding_dim
        self.item_info_dim = 0
        for feat, spec in self.feature_map.features.items():
            if spec.get("source") == "item":
                self.item_info_dim += spec.get("embedding_dim", embedding_dim)

        transformer_in_dim = self.item_info_dim * 2

        self.accumulation_steps = accumulation_steps
        self.embedding_layer = FeatureEmbedding(feature_map, embedding_dim)

        self.transformer_encoder = Transformer(
            transformer_in_dim,
            dim_feedforward=dim_feedforward,
            num_heads=num_heads,
            dropout=transformer_dropout,
            transformer_layers=transformer_layers,
            first_k_cols=first_k_cols,
            concat_max_pool=concat_max_pool
        )
        seq_out_dim = (first_k_cols + int(concat_max_pool)) * transformer_in_dim

        dcn_in_dim = feature_map.sum_emb_out_dim() + seq_out_dim
        self.crossnet = CrossNetV2(dcn_in_dim, dcn_cross_layers)
        self.parallel_dnn = MLP_Block(input_dim=dcn_in_dim,
                                      output_dim=None,  # output hidden layer
                                      hidden_units=dcn_hidden_units,
                                      hidden_activations=hidden_activations,
                                      output_activation=None,
                                      dropout_rates=net_dropout,
                                      batch_norm=batch_norm)
        dcn_out_dim = dcn_in_dim + dcn_hidden_units[-1]
        self.mlp = MLP_Block(input_dim=dcn_out_dim,
                             output_dim=1,
                             hidden_units=mlp_hidden_units,
                             hidden_activations=hidden_activations,
                             output_activation=self.output_activation)
        self.compile(kwargs["optimizer"], kwargs["loss"], learning_rate)
        self.reset_parameters()
        self.model_to_device()

    def forward(self, inputs):
        batch_dict, item_dict, mask = self.get_inputs(inputs)
        
        # 1. Handle user/context features (2D: [batch, total_dim])
        emb_list = []
        if batch_dict:
            feature_emb = self.embedding_layer(batch_dict, flatten_emb=True)
            emb_list.append(feature_emb)
        feat_emb = torch.cat(emb_list, dim=-1)

        # 2. Handle item features manually to avoid the 2D/3D conflict
        # Instead of calling self.embedding_layer(item_dict, flatten_emb=True)
        item_emb_dict = self.embedding_layer.embedding_layer(item_dict)
        
        processed_item_embs = []
        for feat_name, emb in item_emb_dict.items():
            if emb.dim() == 3: # It's a sequence feature like item_tags
                # Mean pool the sequence dimension (dim 1)
                emb = torch.mean(emb, dim=1)
            processed_item_embs.append(emb)
        
        # Now all are 2D: [batch * (seq_len + 1), item_info_dim]
        item_feat_emb = torch.cat(processed_item_embs, dim=-1)
        
        # 3. Reshape and proceed with Transformer
        batch_size = mask.shape[0]
        item_feat_emb = item_feat_emb.view(batch_size, -1, self.item_info_dim)

        target_emb = item_feat_emb[:, -1, :]
        sequence_emb = item_feat_emb[:, 0:-1, : ]
        
        transformer_emb = self.transformer_encoder(target_emb, sequence_emb, mask=mask)

        dcn_in_emb = torch.cat([feat_emb, target_emb, transformer_emb], dim=-1)
        cross_out = self.crossnet(dcn_in_emb)
        dnn_out = self.parallel_dnn(dcn_in_emb)
        y_pred = self.mlp(torch.cat([cross_out, dnn_out], dim=-1))
        
        return {"y_pred": y_pred}

    def get_inputs(self, inputs, feature_source=None):
        batch_dict, item_dict, mask = inputs
        X_dict = dict()
        for feature, value in batch_dict.items():
            if feature in self.feature_map.labels:
                continue
            feature_spec = self.feature_map.features[feature]
            if feature_spec["type"] == "meta":
                continue
            if feature_source and not_in_whitelist(feature_spec["source"], feature_source):
                continue
            X_dict[feature] = value.to(self.device)
        for item, value in item_dict.items():
            item_dict[item] = value.to(self.device)
        return X_dict, item_dict, mask.to(self.device)

    def concat_embedding(self, field, feature_emb_dict):
        if type(field) == tuple:
            emb_list = [feature_emb_dict[f] for f in field]
            return torch.cat(emb_list, dim=-1)
        else:
            return feature_emb_dict[field]

    def get_labels(self, inputs):
        labels = self.feature_map.labels
        batch_dict = inputs[0]
        y = batch_dict[labels[0]].to(self.device)
        return y.float().view(-1, 1)

    def get_group_id(self, inputs):
        return inputs[0][self.feature_map.group_id]

    def train_step(self, batch_data):
        return_dict = self.forward(batch_data)
        y_true = self.get_labels(batch_data)
        loss = self.compute_loss(return_dict, y_true)
        loss = loss / self.accumulation_steps
        loss.backward()
        if (self._batch_index + 1) % self.accumulation_steps == 0:
            nn.utils.clip_grad_norm_(self.parameters(), self._max_gradient_norm)
            self.optimizer.step()
            self.optimizer.zero_grad()
        return loss


class Transformer(nn.Module):
    def __init__(self,
                 transformer_in_dim,
                 dim_feedforward=64,
                 num_heads=1,
                 dropout=0,
                 transformer_layers=1,
                 first_k_cols=16,
                 concat_max_pool=True):
        super(Transformer, self).__init__()
        self.concat_max_pool = concat_max_pool
        self.first_k_cols = first_k_cols
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_in_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=transformer_layers
        )
        if self.concat_max_pool:
            self.out_linear = nn.Linear(transformer_in_dim, transformer_in_dim)

    def forward(self, target_emb, sequence_emb, mask=None):
        # concat action sequence emb with target emb
        seq_len = sequence_emb.size(1)
        concat_seq_emb = torch.cat([sequence_emb,
                                    target_emb.unsqueeze(1).expand(-1, seq_len, -1)], dim=-1)
        # get sequence mask (1's are masked)
        key_padding_mask = self.adjust_mask(mask).bool()  # keep the last dim
        tfmr_out = self.transformer_encoder(src=concat_seq_emb,
                                            src_key_padding_mask=key_padding_mask)
        tfmr_out = tfmr_out.masked_fill(
            key_padding_mask.unsqueeze(-1).repeat(1, 1, tfmr_out.shape[-1]), 0.
        )
        # process the transformer output
        output_concat = []
        output_concat.append(tfmr_out[:, -self.first_k_cols:].flatten(start_dim=1))
        if self.concat_max_pool:
            # Apply max pooling to the transformer output
            tfmr_out = tfmr_out.masked_fill(
                key_padding_mask.unsqueeze(-1).repeat(1, 1, tfmr_out.shape[-1]), -1e9
            )
            pooled_out = self.out_linear(tfmr_out.max(dim=1).values)
            output_concat.append(pooled_out)
        return torch.cat(output_concat, dim=-1)

    def adjust_mask(self, mask):
        # make sure not all actions in the sequence are masked
        fully_masked = mask.all(dim=-1)
        mask[fully_masked, -1] = 0
        return mask

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.dataloader import default_collate
import pandas as pd
import torch


class ParquetDataset(Dataset):
    def __init__(self, data_path):
        self.column_index = dict()
        self.darray = self.load_data(data_path)
        
    def __getitem__(self, index):
        return self.darray[index, :]
    
    def __len__(self):
        return self.darray.shape[0]

    def load_data(self, data_path):
        df = pd.read_parquet(data_path)
        data_arrays = []
        idx = 0
        for col in df.columns:
            if df[col].dtype == "object":
                array = np.array(df[col].to_list())
                seq_len = array.shape[1]
                self.column_index[col] = [i + idx for i in range(seq_len)]
                idx += seq_len
            else:
                array = df[col].to_numpy()
                self.column_index[col] = idx
                idx += 1
            data_arrays.append(array)
        return np.column_stack(data_arrays)


class MMCTRDataLoader(DataLoader):
    def __init__(self, feature_map, data_path, item_info, batch_size=32, shuffle=False,
                 num_workers=1, max_len=100, **kwargs):
        if not data_path.endswith(".parquet"):
            data_path += ".parquet"
        self.dataset = ParquetDataset(data_path)
        column_index = self.dataset.column_index
        super().__init__(dataset=self.dataset, batch_size=batch_size,
                         shuffle=shuffle, num_workers=num_workers,
                         collate_fn=BatchCollator(feature_map, max_len, column_index, item_info))
        self.num_samples = len(self.dataset)
        self.num_blocks = 1
        self.num_batches = int(np.ceil(self.num_samples / self.batch_size))

    def __len__(self):
        return self.num_batches


class BatchCollator(object):
    def __init__(self, feature_map, max_len, column_index, item_info):
        self.feature_map = feature_map
        self.item_info = pd.read_parquet(item_info)
        self.max_len = max_len
        self.column_index = column_index

    def __call__(self, batch):
        batch_tensor = default_collate(batch)
        all_cols = set(list(self.feature_map.features.keys()) + self.feature_map.labels)
        batch_dict = dict()
        for col, idx in self.column_index.items():
            if col in all_cols:
                batch_dict[col] = batch_tensor[:, idx]
        batch_seqs = batch_dict["item_seq"][:, -self.max_len:]
        del batch_dict["item_seq"]
        mask = (batch_seqs > 0).float() # zeros for masked positions
        item_index = batch_dict["item_id"].numpy().reshape(-1, 1)
        del batch_dict["item_id"]
        batch_items = np.hstack([batch_seqs.numpy(), item_index]).flatten()
        item_info = self.item_info.iloc[batch_items]
        item_dict = dict()
        for col in item_info.columns:
            if col in all_cols:
                item_dict[col] = torch.from_numpy(np.array(item_info[col].to_list()))
        return batch_dict, item_dict, mask

## Train model on new embeddings

In [ ]:
import yaml
import os
from fuxictr.preprocess import FeatureMap
from fuxictr.utils import set_logger, print_to_json # Added imports
import logging

# Load configurations
with open("config/dataset_config/MicroLens_dataset.yaml", "r") as f:
    ds_config = yaml.safe_load(f)["MicroLens_1M_x1"]

with open("config/model_config/transformer_dcn_config.yaml", "r") as f:
    full_m_config = yaml.safe_load(f)
    m_config = full_m_config["Transformer_DCN_MicroLens"]

# Set up Logger
# This creates the log directory and initializes the logger for the current model run
log_dir = os.path.join(m_config["model_root"], m_config["model"])
set_logger(m_config) 

# Initialize FeatureMap
feature_map = FeatureMap("MicroLens_1M_x1", ds_config["data_root"])
feature_map.features = {feat["name"]: feat for feat in ds_config["feature_cols"]}
feature_map.labels = [ds_config["label_col"]["name"]]

for name, spec in feature_map.features.items():
    if "embedding_dim" not in spec:
        spec["embedding_dim"] = m_config.get("embedding_dim", 64)

# Log the configurations for reproducibility
logging.info("Params : "+print_to_json(m_config))

# Initialize Model and Data
model = Transformer_DCN(feature_map, **m_config)

item_info_path = "./data/MicroLens_1M_x1/item_info.parquet"
train_gen = MMCTRDataLoader(feature_map, ds_config["train_data"], item_info_path, 
                            batch_size=m_config["batch_size"], shuffle=True)
valid_gen = MMCTRDataLoader(feature_map, ds_config["valid_data"], item_info_path, 
                            batch_size=m_config["batch_size"], shuffle=False)

# 5. Fit Model (Now logs automatically via the initialized logger)
model.fit(train_gen, validation_data=valid_gen, epochs=m_config["epochs"])

Running 1st-place model on training ...
2025-12-20 02:15:24,952 P67 INFO FuxiCTR version: 2.3.7
2025-12-20 02:15:24,952 P67 INFO Params: {
    "batch_norm": "True",
    "batch_size": "1024",
    "concat_max_pool": "True",
    "data_format": "parquet",
    "data_root": "./data/",
    "dataset_id": "MicroLens_1M_x1",
    "dcn_cross_layers": "3",
    "dcn_hidden_units": "[512, 256, 128]",
    "debug_mode": "False",
    "dim_feedforward": "512",
    "early_stop_patience": "5",
    "embedding_dim": "64",
    "embedding_regularizer": "0",
    "epochs": "15",
    "eval_steps": "None",
    "feature_cols": "[{'active': True, 'dtype': 'int', 'name': 'user_id', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'item_seq', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'likes_level', 'type': 'categorical', 'vocab_size': 11}, {'active': True, 'dtype': 'int', 'name': 'views_level', 'type': 'categorical', 'vocab_size': 11}, {'active': True, 'dtype': 'int', 'name': 'item_id', 'sourc

## Test the model using test.parquet (generates prediction.csv to sumbit)

In [ ]:
import logging
import os
import shutil
import pandas as pd

# Use the model_id or a custom string for the archive name
experiment_id = m_config.get("model_id", "Transformer_DCN_v1")

logging.info('Running 1st-place model on test set...')

# 1. Use MMCTRDataLoader to match your custom data processing logic
test_gen = MMCTRDataLoader(feature_map, 
                           ds_config["test_data"], 
                           item_info_path, 
                           batch_size=m_config["batch_size"], 
                           shuffle=False)

# 2. Generate predictions
test_pred = model.predict(test_gen)

# 3. Create the submission DataFrame
# Note: Ensure test_pred is flattened to 1D
ans = pd.DataFrame({
    "ID": range(len(test_pred)),
    "Task1": test_pred.flatten(),
    "Task2": test_pred.flatten() 
})

logging.info("Writing results...")

# 4. Save and archive
os.makedirs("submission", exist_ok=True)
ans.to_csv("submission/prediction.csv", index=False)

# This creates submission/{experiment_id}.zip containing prediction.csv
shutil.make_archive(f'submission/{experiment_id}', 'zip', 'submission/', 'prediction.csv')

logging.info("All done.")

Running 1st-place model on test set...
2025-12-20 12:25:24,799 P463 INFO FuxiCTR version: 2.3.7
2025-12-20 12:25:24,799 P463 INFO Params: {
    "batch_norm": "True",
    "batch_size": "1024",
    "concat_max_pool": "True",
    "data_format": "parquet",
    "data_root": "./data/",
    "dataset_id": "MicroLens_1M_x1",
    "dcn_cross_layers": "3",
    "dcn_hidden_units": "[512, 256, 128]",
    "debug_mode": "False",
    "dim_feedforward": "512",
    "early_stop_patience": "5",
    "embedding_dim": "64",
    "embedding_regularizer": "0",
    "epochs": "15",
    "eval_steps": "None",
    "feature_cols": "[{'active': True, 'dtype': 'int', 'name': 'user_id', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'item_seq', 'type': 'meta'}, {'active': True, 'dtype': 'int', 'name': 'likes_level', 'type': 'categorical', 'vocab_size': 11}, {'active': True, 'dtype': 'int', 'name': 'views_level', 'type': 'categorical', 'vocab_size': 11}, {'active': True, 'dtype': 'int', 'name': 'item_id', 'sour